# Experiment 2: Email Spam/Ham Classification
This standalone notebook implements text vectorization, Naive Bayes, K-Nearest Neighbors, and Logistic Regression with comprehensive performance evaluation.

In [1]:
import os
import matplotlib
matplotlib.use('Agg') # Strictly headless - non-interfering, zero GUI popups

def resolve_path(rel_path):
    """Dynamically resolves datasets whether run from repo root or Ex subfolder."""
    for prefix in ['', '../', '../../']:
        cand = os.path.join(prefix, rel_path)
        if os.path.exists(cand):
            return cand
    return rel_path

def resolve_out(rel_path):
    """Avoids nested directories if running from within Ex2."""
    if os.path.basename(os.getcwd()) == 'Ex2':
        if rel_path.startswith('Ex2/'):
            return rel_path[len('Ex2/'):]
    return rel_path

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs(resolve_out('Ex2/plots'), exist_ok=True)

In [2]:
def run_experiment_2(csv_path="Datasets/Email_Spam_Dataset/emails.csv"):
    print("="*60)
    print("=== LAUNCHING EXPERIMENT 2: EMAIL SPAM CLASSIFICATION ===")
    print("="*60)
    
    path = resolve_path(csv_path)
    if not os.path.exists(path):
        path = resolve_path("Ex2/spambase_csv.csv")
    df = pd.read_csv(path)
    
    if 'Email No.' in df.columns:
        df = df.drop(columns=['Email No.'])
    target_col = 'Prediction' if 'Prediction' in df.columns else df.columns[-1]
    
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_te_sc = scaler.transform(X_te)
    
    models = {
        'Multinomial Naive Bayes': MultinomialNB(),
        'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
    }
    
    results = {}
    for name, model in models.items():
        if name == 'Multinomial Naive Bayes':
            model.fit(np.abs(X_tr), y_tr)
            y_pred = model.predict(np.abs(X_te))
        else:
            model.fit(X_tr_sc, y_tr)
            y_pred = model.predict(X_te_sc)
            
        results[name] = {
            'Accuracy': round(accuracy_score(y_te, y_pred) * 100, 2),
            'Precision': round(precision_score(y_te, y_pred, zero_division=0) * 100, 2),
            'Recall': round(recall_score(y_te, y_pred, zero_division=0) * 100, 2),
            'F1-Score': round(f1_score(y_te, y_pred, zero_division=0) * 100, 2)
        }
        
    print("\n=== EXPERIMENT 2 PIPELINE COMPLETE ===")
    return results

In [3]:
# Master Execution Cell
ex2_output = run_experiment_2()
display(pd.DataFrame(ex2_output).T.style.background_gradient(cmap='Blues', subset=['Accuracy', 'F1-Score']))

=== LAUNCHING EXPERIMENT 2: EMAIL SPAM CLASSIFICATION ===



=== EXPERIMENT 2 PIPELINE COMPLETE ===


,Accuracy,Precision,Recall,F1-Score
Multinomial Naive Bayes,94.200000,86.810000,94.330000,90.420000
KNN (k=5),83.380000,64.350000,95.670000,76.940000
Logistic Regression,96.910000,92.410000,97.330000,94.810000
